# Partner Demand Forecaster — EDA + XGBoost Quantile Rebuild
Analyst notes: load export EDA, inspect partner-country pairs, build q10/q50/q90 XGB regressors.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import xgboost as xgb, joblib, json
df = pd.read_csv('backend/brain/data/final_csv/01_partner_discovery_india_as_exporter_eda.csv', nrows=50000)
print('Shape', df.shape)
print(df[['partner_name','hs6','trade_value_usd','year']].head())


## Missingness / target (demand proxy = trade_value_usd)

In [ ]:
missing=df.isnull().mean().sort_values(ascending=False)
print(missing.head())
print('Year range', df['year'].min(), df['year'].max())


## Quantile regression — q50 baseline

In [ ]:
num=['trade_value_usd','net_weight_kg'] if 'net_weight_kg' in df.columns else ['trade_value_usd']
num=[c for c in num if c in df.columns]
X=df[['partner_name','hs6','year']+num].fillna(0)
# encode categoricals quickly via factorize for rebuild demo
X['partner_enc']=X['partner_name'].astype('category').cat.codes
X['hs6_enc']=X['hs6'].astype('category').cat.codes
X=X[['partner_enc','hs6_enc','year']+num]
y=df['trade_value_usd'].fillna(0).values
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
clf=xgb.XGBRegressor(objective='reg:quantileerror',quantile_alpha=0.5,n_estimators=100,max_depth=5,learning_rate=0.05,n_jobs=-1,random_state=42)
clf.fit(Xtr,ytr)
print('MAE', mean_absolute_error(yte,clf.predict(Xte)))


## Save rebuild artifacts

In [ ]:
os.makedirs('backend/brain/models_rebuild/partner_discovery_xgb',exist_ok=True)
joblib.dump(clf,'backend/brain/models_rebuild/partner_discovery_xgb/demand_q50.joblib')
with open('backend/brain/models_rebuild/partner_discovery_xgb/metadata.json','w') as f: json.dump({'type':'XGBRegressor_quantile','rebuild':'2026-08-26'},f)
print('Partner artifacts saved.')


Notes: full dataset 16MB. Notebook uses 50k sample for rebuild speed; production uses full `01_partner_discovery_india_as_exporter_eda.csv`. Artifacts saved to rebuild folder only.